In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔍 Fraud Detection & Anomaly Detection Guide

**Building systems to detect fraud, anomalies, and suspicious patterns in real-time**

This guide covers:
- Anomaly detection algorithms
- Rule-based fraud detection
- Machine learning approaches
- Feature engineering
- Real-time scoring
- False positive management
- Feedback loops and model retraining
- Operational fraud systems
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Anomaly Detection Fundamentals

### Statistical Anomaly Detection
```python
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime, timedelta
import statistics

@dataclass
class AnomalyScore:
    """Anomaly score result"""
    entity_id: str
    score: float  # 0-1, where 1 is most anomalous
    threshold: float
    is_anomaly: bool
    reason: str
    timestamp: datetime = field(default_factory=datetime.now)

class ZScoreAnomalyDetector:
    """Z-score based anomaly detection"""
    
    def __init__(self, threshold: float = 3.0):
        self.threshold = threshold  # Standard deviations
        self.history: Dict[str, List[float]] = {}
        self.detections: List[AnomalyScore] = []
    
    def add_observation(self, entity_id: str, value: float):
        """Add observation to history"""
        
        if entity_id not in self.history:
            self.history[entity_id] = []
        
        self.history[entity_id].append(value)
        
        # Keep only recent observations (e.g., last 100)
        if len(self.history[entity_id]) > 100:
            self.history[entity_id] = self.history[entity_id][-100:]
    
    def is_anomaly(self, entity_id: str, value: float) -> AnomalyScore:
        """Check if value is anomalous"""
        
        if entity_id not in self.history or len(self.history[entity_id]) < 10:
            return AnomalyScore(
                entity_id=entity_id,
                score=0.0,
                threshold=self.threshold,
                is_anomaly=False,
                reason="Insufficient history"
            )
        
        values = self.history[entity_id]
        mean = statistics.mean(values)
        stdev = statistics.stdev(values) if len(values) > 1 else 0
        
        if stdev == 0:
            return AnomalyScore(
                entity_id=entity_id,
                score=0.0,
                threshold=self.threshold,
                is_anomaly=False,
                reason="No variance in history"
            )
        
        z_score = abs((value - mean) / stdev)
        
        is_anomaly = z_score > self.threshold
        
        # Normalize score to 0-1
        normalized_score = min(1.0, z_score / self.threshold)
        
        result = AnomalyScore(
            entity_id=entity_id,
            score=normalized_score,
            threshold=self.threshold,
            is_anomaly=is_anomaly,
            reason=f"Z-score: {z_score:.2f}"
        )
        
        if is_anomaly:
            self.detections.append(result)
        
        return result

class IQROutlierDetector:
    """Interquartile range based outlier detection"""
    
    def __init__(self, multiplier: float = 1.5):
        self.multiplier = multiplier
        self.history: Dict[str, List[float]] = {}
    
    def is_outlier(self, entity_id: str, value: float) -> AnomalyScore:
        """Check if value is outlier"""
        
        if entity_id not in self.history or len(self.history[entity_id]) < 4:
            return AnomalyScore(
                entity_id=entity_id,
                score=0.0,
                threshold=0.0,
                is_anomaly=False,
                reason="Insufficient history"
            )
        
        sorted_values = sorted(self.history[entity_id])
        n = len(sorted_values)
        
        q1 = sorted_values[n // 4]
        q3 = sorted_values[3 * n // 4]
        iqr = q3 - q1
        
        lower_bound = q1 - self.multiplier * iqr
        upper_bound = q3 + self.multiplier * iqr
        
        is_outlier = value < lower_bound or value > upper_bound
        
        if is_outlier:
            distance = min(abs(value - lower_bound), abs(value - upper_bound))
            score = min(1.0, distance / max(1.0, iqr))
        else:
            score = 0.0
        
        return AnomalyScore(
            entity_id=entity_id,
            score=score,
            threshold=0.0,
            is_anomaly=is_outlier,
            reason=f"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]"
        )
```

### Isolation Forest Algorithm (Simplified)
```python
class IsolationForestDetector:
    """Isolation forest for anomaly detection"""
    
    def __init__(self, num_trees: int = 100, sample_size: int = 256):
        self.num_trees = num_trees
        self.sample_size = sample_size
        self.trees: List[Dict] = []
        self.training_data: List[Dict] = []
    
    def train(self, data: List[Dict], feature_names: List[str]):
        """Train isolation forest"""
        
        self.training_data = data
        
        for _ in range(self.num_trees):
            # Sample data
            sample = self._random_sample(data, self.sample_size)
            
            # Build tree
            tree = self._build_tree(sample, feature_names, depth=0)
            self.trees.append(tree)
    
    def _build_tree(self, data: List[Dict], feature_names: List[str], depth: int) -> Dict:
        """Recursively build isolation tree"""
        
        if len(data) <= 1 or depth >= 10:  # Max depth
            return {'type': 'leaf', 'size': len(data)}
        
        # Randomly select feature
        feature = feature_names[len(feature_names) % (depth + 1)] if feature_names else None
        
        if not feature:
            return {'type': 'leaf', 'size': len(data)}
        
        # Randomly select split value
        values = [d.get(feature, 0) for d in data]
        split_value = statistics.median(values)
        
        left = [d for d in data if d.get(feature, 0) <= split_value]
        right = [d for d in data if d.get(feature, 0) > split_value]
        
        return {
            'type': 'node',
            'feature': feature,
            'split_value': split_value,
            'left': self._build_tree(left, feature_names, depth + 1),
            'right': self._build_tree(right, feature_names, depth + 1)
        }
    
    def _random_sample(self, data: List[Dict], size: int) -> List[Dict]:
        """Random sample from data"""
        
        import random
        
        return random.sample(data, min(size, len(data)))
    
    def predict_anomaly(self, sample: Dict, feature_names: List[str]) -> AnomalyScore:
        """Predict if sample is anomalous"""
        
        anomaly_scores = []
        
        for tree in self.trees:
            depth = self._get_depth(tree, sample, feature_names)
            anomaly_scores.append(depth)
        
        avg_depth = sum(anomaly_scores) / len(anomaly_scores) if anomaly_scores else 0
        
        # Normalize to 0-1
        max_depth = 10
        anomaly_score = avg_depth / max_depth
        
        return AnomalyScore(
            entity_id=str(sample),
            score=min(1.0, anomaly_score),
            threshold=0.5,
            is_anomaly=anomaly_score > 0.5,
            reason=f"Average depth: {avg_depth:.2f}"
        )
    
    def _get_depth(self, node: Dict, sample: Dict, feature_names: List[str], depth: int = 0) -> int:
        """Get depth of sample in tree"""
        
        if node.get('type') == 'leaf':
            return depth
        
        feature = node.get('feature')
        split_value = node.get('split_value')
        
        if sample.get(feature, 0) <= split_value:
            return self._get_depth(node['left'], sample, feature_names, depth + 1)
        else:
            return self._get_depth(node['right'], sample, feature_names, depth + 1)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Fraud Detection Systems

### Rule-Based Fraud Detection
```python
@dataclass
class FraudRule:
    """Fraud detection rule"""
    rule_id: str
    name: str
    condition: callable
    severity: str  # low, medium, high, critical
    enabled: bool = True

class RuleBasedFraudDetector:
    """Rule-based fraud detection"""
    
    def __init__(self):
        self.rules: Dict[str, FraudRule] = {}
        self.detections: List[Dict] = []
    
    def add_rule(self, rule: FraudRule):
        """Add fraud rule"""
        self.rules[rule.rule_id] = rule
    
    def check_transaction(self, transaction: Dict) -> List[Dict]:
        """Check transaction against rules"""
        
        triggered_rules = []
        
        for rule_id, rule in self.rules.items():
            if not rule.enabled:
                continue
            
            try:
                if rule.condition(transaction):
                    triggered_rules.append({
                        'rule_id': rule_id,
                        'rule_name': rule.name,
                        'severity': rule.severity,
                        'timestamp': datetime.now()
                    })
                    
            except Exception as e:
                print(f"Error evaluating rule {rule_id}: {e}")
        
        if triggered_rules:
            self.detections.append({
                'transaction': transaction,
                'rules_triggered': triggered_rules,
                'detected_at': datetime.now()
            })
        
        return triggered_rules

class FraudScoringEngine:
    """Calculate fraud score"""
    
    def __init__(self):
        self.rule_weights: Dict[str, float] = {}
    
    def set_rule_weight(self, rule_id: str, weight: float):
        """Set weight for rule"""
        self.rule_weights[rule_id] = weight
    
    def calculate_fraud_score(self, triggered_rules: List[Dict]) -> float:
        """Calculate fraud score from triggered rules"""
        
        total_score = 0.0
        
        for rule in triggered_rules:
            weight = self.rule_weights.get(rule['rule_id'], 1.0)
            
            if rule['severity'] == 'critical':
                score_contrib = 0.4 * weight
            elif rule['severity'] == 'high':
                score_contrib = 0.3 * weight
            elif rule['severity'] == 'medium':
                score_contrib = 0.2 * weight
            else:
                score_contrib = 0.1 * weight
            
            total_score += score_contrib
        
        return min(1.0, total_score)
```

### Velocity & Pattern Analysis
```python
class VelocityDetector:
    """Detect suspicious velocity patterns"""
    
    def __init__(self, time_window_minutes: int = 60):
        self.time_window = timedelta(minutes=time_window_minutes)
        self.entity_events: Dict[str, List[datetime]] = {}
    
    def record_event(self, entity_id: str):
        """Record event for entity"""
        
        now = datetime.now()
        
        if entity_id not in self.entity_events:
            self.entity_events[entity_id] = []
        
        self.entity_events[entity_id].append(now)
        
        # Clean up old events
        cutoff = now - self.time_window
        self.entity_events[entity_id] = [t for t in self.entity_events[entity_id] if t > cutoff]
    
    def get_event_velocity(self, entity_id: str, threshold: int = 10) -> Tuple[int, bool]:
        """Get event velocity and check if anomalous"""
        
        if entity_id not in self.entity_events:
            return 0, False
        
        event_count = len(self.entity_events[entity_id])
        is_suspicious = event_count > threshold
        
        return event_count, is_suspicious
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Model Feedback & Retraining

### Feedback Collection
```python
class FraudFeedbackCollector:
    """Collect feedback on fraud predictions"""
    
    def __init__(self):
        self.feedback: List[Dict] = []
        self.false_positives: int = 0
        self.false_negatives: int = 0
    
    def record_feedback(self, prediction_id: str, actual_label: str, predicted_label: str):
        """Record feedback for prediction"""
        
        is_correct = actual_label == predicted_label
        
        if not is_correct:
            if predicted_label == "fraud" and actual_label == "legitimate":
                self.false_positives += 1
            elif predicted_label == "legitimate" and actual_label == "fraud":
                self.false_negatives += 1
        
        self.feedback.append({
            'prediction_id': prediction_id,
            'actual': actual_label,
            'predicted': predicted_label,
            'correct': is_correct,
            'timestamp': datetime.now()
        })
    
    def get_model_metrics(self) -> Dict:
        """Get model performance metrics"""
        
        total = len(self.feedback)
        correct = sum(1 for f in self.feedback if f['correct'])
        
        accuracy = correct / total if total > 0 else 0
        
        return {
            'total_predictions': total,
            'accuracy': accuracy,
            'false_positives': self.false_positives,
            'false_negatives': self.false_negatives,
            'precision': self._calculate_precision(),
            'recall': self._calculate_recall()
        }
    
    def _calculate_precision(self) -> float:
        """Calculate precision"""
        
        fraud_predictions = sum(1 for f in self.feedback if f['predicted'] == 'fraud')
        correct_fraud = sum(1 for f in self.feedback if f['predicted'] == 'fraud' and f['correct'])
        
        return correct_fraud / fraud_predictions if fraud_predictions > 0 else 0
    
    def _calculate_recall(self) -> float:
        """Calculate recall"""
        
        actual_fraud = sum(1 for f in self.feedback if f['actual'] == 'fraud')
        detected_fraud = sum(1 for f in self.feedback if f['actual'] == 'fraud' and f['correct'])
        
        return detected_fraud / actual_fraud if actual_fraud > 0 else 0
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Fraud & Anomaly Detection Checklist

✅ **Detection Methods**
- [ ] Baseline statistics calculated
- [ ] Z-score detection working
- [ ] Isolation forest trained
- [ ] Rule engine implemented
- [ ] Threshold tuned

✅ **Fraud Rules**
- [ ] Velocity checks enabled
- [ ] Pattern detection
- [ ] Amount thresholds
- [ ] Geolocation checks
- [ ] Velocity scoring

✅ **Real-Time Scoring**
- [ ] Scoring engine deployed
- [ ] Rule weights calibrated
- [ ] Response time <100ms
- [ ] Fallback strategy
- [ ] Escalation logic

✅ **False Positive Management**
- [ ] Feedback collection
- [ ] Precision tracking
- [ ] Recall tracking
- [ ] Customer experience
- [ ] Allowlisting

✅ **Model & Operations**
- [ ] Data pipeline
- [ ] Model retraining
- [ ] A/B testing
- [ ] Monitoring dashboard
- [ ] Incident response
</VSCode.Cell>
```